# v4 — Remove mislabelled reviews, then retrain
Cross-fitted confident learning: each review is judged by a model that never saw it.
The val/test sets are never cleaned, so v3 vs v4 stays fair. Runtime → T4 GPU → Run all. ~2 h.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Checkpoints go to Google Drive so a disconnect can be resumed. If the mount fails
# ("credential propagation was unsuccessful" happens with several Google accounts in one browser),
# fall back to the VM's disk: training still works, but a disconnect restarts from zero.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v4_label_noise_cleaning'
except Exception as e:
    print(f'Drive mount failed ({e}); checkpoints stay on the VM.')
    DRIVE = '/content/btp_v4_label_noise_cleaning'
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
RESULTS = 'versions/v4_label_noise_cleaning/results'
!pip -q install -U transformers datasets accelerate scikit-learn

In [ ]:
# ~8 min: identical 5-class splits to v3
!python prepare_data.py --classes 5 --per-class 40000

In [ ]:
# ~35 min: 2 folds x 1 epoch, out-of-sample predictions, drop confident contradictions
!python detect_label_noise.py --data-dir data5 --fold-epochs 1 --batch-size 32 --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{RESULTS}/label_noise.json')), indent=2))
pd.set_option('display.max_colwidth', 110)
display(pd.read_csv(f'{RESULTS}/errors/label_noise_sample.csv').head(15))

In [ ]:
# ~1-1.5 h: retrain on the cleaned training set, scored on the UNCHANGED test sets
!python finetune_roberta.py --name roberta_base_5class_cleaned --data-dir data5_clean --batch-size 32 --eval-steps 2500 --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt

In [ ]:
import os
assert os.path.exists('models/roberta_base_5class_cleaned/config.json'), 'Training failed: scroll up. Re-run the cell to resume from the checkpoint.'
print('Training finished OK')

In [ ]:
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for k in ['roberta_base_5class_cleaned', 'roberta_base_5class_cleaned_natural', 'roberta_base_5class_cleaned_natural_prior']:
    if k in m:
        v = m[k]
        print(f"{k:28s} acc {v['accuracy']:.4f}  macro-F1 {v['macro_f1']:.4f}  "
              f"off-by-one {v['off_by_one_accuracy']:.4f}  MAE {v['mae_classes']:.3f}  QWK {v['quadratic_weighted_kappa']:.4f}")

In [ ]:
!rm -rf models/fold_a models/fold_b

In [ ]:
!zip -qr btp_v4_label_noise_cleaning.zip models/roberta_base_5class_cleaned versions/v4_label_noise_cleaning/results && ls -lh btp_v4_label_noise_cleaning.zip
!cp btp_v4_label_noise_cleaning.zip {DRIVE}/
from google.colab import files
files.download('btp_v4_label_noise_cleaning.zip')